Simulator: Multiple Sources
===========================

This script simulates `PointDataset` data of a strong lens which has multiple lensed point sources at different
redshifts (the example below has two sources, but the same approach extends to any number).

The simulated system is a multi-plane lens where source_0 is itself a deflector for source_1 behind it: a
foreground galaxy lens (z=0.5) lenses both background sources, while source_0's own mass at z=1.0 additionally
lenses source_1 at z=2.0. The result is two interleaved sets of multiple images (a "double Einstein cross"-like
configuration). Each source's multiple images are stored as a separate `PointDataset`, and the companion
`modeling.py` script fits both datasets jointly using the multi-dataset/factor-graph API.

The lens model below uses:

 - The foreground lens galaxy's total mass distribution is an `Isothermal`.
 - The first source `Galaxy` (at z=1.0) is itself a `Galaxy` with a mass profile and a `Point`.
 - The second source `Galaxy` (at z=2.0) is a `Point`-only galaxy.

__Currently Blocked By PyAutoLens #480__

This script does not run end-to-end on the current PyAutoLens release. The `PointSolver` magnification filter
uses the tracer's last-plane magnification instead of the requested `plane_redshift`'s magnification, so every
candidate image position for source_0 (which is at the intermediate plane z=1.0) is rejected and the solver
returns 0 positions. See https://github.com/PyAutoLabs/PyAutoLens/issues/480 — the simulator and modeling scripts
in this folder are listed in `config/build/no_run.yaml` until that bug is fixed. The script is left here in its
intended form so the example is correct as soon as #480 lands; no script changes will be needed once it does.

__Contents__

- **Dataset Paths:** The `dataset_type` describes the type of data being simulated and `dataset_name` gives it a.
- **Ray Tracing:** Setup the lens galaxy's mass (SIE) and source galaxy `Point` for this simulated lens.
- **Point Solver:** We use a `PointSolver` to locate the multiple images.
- **Point Datasets:** Create a point-source data object and output this to a `.json` file, which is the format used to.
- **Visualize:** Output a subplot of the simulated point source dictionary and the tracer's quantities to the.
- **Tracer json:** Save the `Tracer` in the dataset folder as a .json file, ensuring the true light profiles, mass.

__Start Here Notebook__

If any code in this script is unclear, refer to the `point_source/simulator.ipynb` notebook.

__Google Colab Setup__

Set up Google Colab as in [the introductory notebook](https://github.com/PyAutoLabs/autolens_workspace/blob/main/start_here.ipynb).


In [ ]:
# @title Set up Google Colab { display-mode: "form" }
try:
    import google.colab
except ImportError:
    from autolens import setup_colab as _setup_colab
else:
    import importlib
    import subprocess
    import sys

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "autonerves", "--no-deps"]
    )
    _setup_colab = importlib.import_module("autonerves.setup_colab")

_setup_colab.setup("autolens")

In [ ]:

from autolens import jax_wrapper  # Sets JAX environment before other imports

from autolens import setup_notebook; setup_notebook()

from pathlib import Path
import numpy as np
import autolens as al
import autolens.plot as aplt

__Dataset Paths__

The `dataset_type` describes the type of data being simulated and `dataset_name` gives it a descriptive name. 

In [ ]:
dataset_type = "point_source"
dataset_name = "multiple_sources"

The path where the dataset will be output.

In [ ]:
dataset_path = Path("dataset") / dataset_type / dataset_name

__Ray Tracing__

Setup the lens galaxy's mass (SIE) and source galaxy `Point` for this simulated lens. We include a
faint disk in the source for purely visualisation purposes to show where the multiple images appear.

No external shear is simulated here. The modeling example still fits one, composed as an `ExternalShear` in a
`MassField` in the model's `fields` slot (see `imaging/modeling.py`), so its inferred `gamma_1` and
`gamma_2` should be consistent with zero.

For lens modeling, defining ellipticity in terms of the `ell_comps` improves the model-fitting procedure. However,
for simulating a strong lens you may find it more intuitive to define the elliptical geometry using the
axis-ratio of the profile (axis_ratio = semi-major axis / semi-minor axis = b/a) and position angle, where angle
is in degrees and defined counter clockwise from the positive x-axis.

We can use the `convert` module to determine the elliptical components from the axis-ratio and angle.

In [ ]:
lens_galaxy = al.Galaxy(
    redshift=0.5,
    mass=al.mp.Isothermal(
        centre=(0.0, 0.0),
        einstein_radius=1.6,
        ell_comps=al.convert.ell_comps_from(axis_ratio=0.9, angle=45.0),
    ),
)

source_galaxy_0 = al.Galaxy(
    redshift=1.0,
    mass=al.mp.Isothermal(
        centre=(0.02, 0.03),
        einstein_radius=0.2,
        ell_comps=al.convert.ell_comps_from(axis_ratio=0.8, angle=60.0),
    ),
    light=al.lp.ExponentialCore(
        centre=(0.02, 0.03), intensity=0.1, effective_radius=0.02
    ),
    point_0=al.ps.Point(centre=(0.02, 0.03)),
)


source_galaxy_1 = al.Galaxy(
    redshift=2.0,
    light=al.lp.ExponentialCore(
        centre=(0.0, 0.0), intensity=0.1, effective_radius=0.02
    ),
    point_1=al.ps.Point(centre=(0.0, 0.0)),
)

Use these galaxies to setup a tracer, which will compute the multiple image positions of the simulated dataset.

In [ ]:
tracer = al.Tracer(galaxies=[lens_galaxy, source_galaxy_0, source_galaxy_1])

__Point Solver__

We use a `PointSolver` to locate the multiple images. 

In [ ]:
grid = al.Grid2D.uniform(
    shape_native=(200, 200),
    pixel_scales=0.05,  # <- The pixel-scale describes the conversion from pixel units to arc-seconds.
)

solver = al.PointSolver.for_grid(
    grid=grid, pixel_scale_precision=0.001, magnification_threshold=0.1
)

We now pass the `Tracer` to the solver.

This finds the image-plane coordinates that map directly to the source-plane centres (0.02", 0.03") and
(0.0", 0.0"). A double Einstein cross is a multi-plane lensing system, therefore for each source we also pass
their redshift into the solver as `plane_redshift` so that it finds the multiple images while properly accounting
for the multi-plane lensing.

Position noise is set to 0.005" (5 mas, realistic PSF-centroiding precision on HST imaging) and flux noise
to 5% relative (microlensing-dominated regime). See `scripts/point_source/simulator.py` for a full
discussion of these values.

In [ ]:
position_noise = 0.005
flux_rel_noise = 0.05

positions_0 = solver.solve(
    tracer=tracer,
    source_plane_coordinate=source_galaxy_0.point_0.centre,
    plane_redshift=source_galaxy_0.redshift,
)

positions_0_with_noise = positions_0 + np.random.normal(
    loc=0.0, scale=position_noise, size=positions_0.shape
)

positions_0_with_noise = al.Grid2DIrregular(
    values=positions_0_with_noise,
)

positions_1 = solver.solve(
    tracer=tracer,
    source_plane_coordinate=source_galaxy_1.point_1.centre,
    plane_redshift=source_galaxy_1.redshift,
)

positions_1_with_noise = positions_1 + np.random.normal(
    loc=0.0, scale=position_noise, size=positions_1.shape
)

positions_1_with_noise = al.Grid2DIrregular(
    values=positions_1_with_noise,
)

Use the positions to compute the magnification of the `Tracer` at every position.

In [ ]:
magnifications_0 = al.LensCalc.from_tracer(
    tracer=tracer
).magnification_2d_via_hessian_from(grid=positions_0)
magnifications_1 = al.LensCalc.from_tracer(
    tracer=tracer
).magnification_2d_via_hessian_from(grid=positions_1)

We can now compute the observed fluxes of the `Point`, give we know how much each is magnified.

In [ ]:
flux = 1.0
fluxes_0 = [flux * np.abs(magnification) for magnification in magnifications_0]
fluxes_0 = al.ArrayIrregular(values=fluxes_0)
fluxes_0_with_noise = fluxes_0 + np.random.normal(
    loc=0.0, scale=flux_rel_noise * np.asarray(fluxes_0), size=len(fluxes_0)
)
fluxes_0_noise_map = al.ArrayIrregular(values=flux_rel_noise * np.asarray(fluxes_0))

fluxes_1 = [flux * np.abs(magnification) for magnification in magnifications_1]
fluxes_1 = al.ArrayIrregular(values=fluxes_1)
fluxes_1_with_noise = fluxes_1 + np.random.normal(
    loc=0.0, scale=flux_rel_noise * np.asarray(fluxes_1), size=len(fluxes_1)
)
fluxes_1_noise_map = al.ArrayIrregular(values=flux_rel_noise * np.asarray(fluxes_1))

We now output the image of this strong lens to `.fits` which can be used for visualize when performing point-source 
modeling and to `.png` for general inspection.

In [ ]:

aplt.plot_array(array=tracer.image_2d_from(grid=grid), title="Image")

aplt.subplot_tracer(
    tracer=tracer, grid=grid, output_path=dataset_path, output_format="png"
)
aplt.subplot_galaxies_images(
    tracer=tracer, grid=grid, output_path=dataset_path, output_format="png"
)

__Point Datasets__

Create a point-source data object and output this to a `.json` file, which is the format used to load and
analyse the dataset.

In [ ]:
dataset_0 = al.PointDataset(
    name="point_0",
    positions=positions_0_with_noise,
    positions_noise_map=position_noise,
    fluxes=fluxes_0_with_noise,
    fluxes_noise_map=fluxes_0_noise_map,
)
dataset_1 = al.PointDataset(
    name="point_1",
    positions=positions_1_with_noise,
    positions_noise_map=position_noise,
    fluxes=fluxes_1_with_noise,
    fluxes_noise_map=fluxes_1_noise_map,
)


We now output the point datasets to the dataset path as a .json file, which is loaded in the point source modeling
examples.

In [ ]:
al.output_to_json(
    obj=dataset_0,
    file_path=dataset_path / "point_dataset_0.json",
)

al.output_to_json(
    obj=dataset_1,
    file_path=dataset_path / "point_dataset_1.json",
)

__CSV Output__

Both datasets can also be saved to a single CSV — one row per observed image grouped by
``name`` — as the hand-editable spreadsheet format alternative to per-source JSON.

In [ ]:
al.output_to_csv(
    datasets=[dataset_0, dataset_1],
    file_path=dataset_path / "point_datasets.csv",
)

__Visualize__

Output a subplot of the simulated point source dictionary and the tracer's quantities to the dataset path as .png files.

In [ ]:
aplt.subplot_point_dataset(
    dataset=dataset_0, output_path=dataset_path, output_format="png"
)
aplt.subplot_point_dataset(
    dataset=dataset_1, output_path=dataset_path, output_format="png"
)

aplt.subplot_tracer(
    tracer=tracer, grid=grid, output_path=dataset_path, output_format="png"
)
aplt.subplot_galaxies_images(
    tracer=tracer, grid=grid, output_path=dataset_path, output_format="png"
)

__Tracer json__

Save the `Tracer` in the dataset folder as a .json file, ensuring the true light profiles, mass profiles and galaxies
are safely stored and available to check how the dataset was simulated in the future. 

This can be loaded via the method `tracer = al.from_json()`.

In [ ]:
al.output_to_json(
    obj=tracer,
    file_path=dataset_path / "tracer.json",
)
